In [21]:
import pandas as pd

df = pd.read_csv('df_aggiornato.csv')

In [22]:
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied,Year,Month,Day_name
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,2024,4,Monday
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,2023,7,Sunday
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,2022,10,Wednesday
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN,2022,5,Saturday
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,2022,10,Sunday


In [23]:
#Valori mancanti  
missing= df.isna().sum().to_frame('n_missing')
missing['%_missing']= (missing['n_missing']/len(df)*100).round(2)
missing.sort_values('n_missing',ascending=False)

,n_missing,%_missing
Discount Applied,4199,33.39
Item,1213,9.65
Price Per Unit,609,4.84
Quantity,604,4.80
Total Spent,604,4.80
Transaction ID,0,0.00
Category,0,0.00
Customer ID,0,0.00
Location,0,0.00
Payment Method,0,0.00


Per gestire i valori mancanti in questo dataset , analizzo le relazioni logiche e matematiche tra le colonne 
1) Esiste una relazione vincolante tra le colonne :
**Total Spent = Price Per Unit X Quantity**

Quindi calcolo :

**Total Spent** --> Price Per Unit X Quantity

**Price Per Unit** --> Total Spent / Quantity

**Quantity** --> Total Spent / Price Per Unità

2) Tra il prodotto e la sua categoria/prezzo c'è un legame 
Per trovare i valori di **Item** , se conosco il Price Per Unit e la Category, posso verificare nel resto del dataset se esiste un unico prodotto appartenente a quella categoria con esattamente quel prezzo unitario. In caso positivo, posso risalire al nome dell'articolo mancante.

 3) La colonna **Discount Applied** contiene valori booleani mancanti , "NaN" ma potrebbero anche significare che non sono stati applicati sconti. Posso Confrontare se la colonna  Total Spent è uguale alla colonna 'Total Spent calcololato ' che creo in questo modo :Price Per Unit X Quantity 

In [24]:
df.columns

Index(['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit',
       'Quantity', 'Total Spent', 'Payment Method', 'Location',
       'Transaction Date', 'Discount Applied', 'Year', 'Month', 'Day_name'],
      dtype='str')

**Verifica 1**

In [25]:
#Calcolo di 'Quantity' dove ho  Total Spent e Price Per Unit
df['Quantity'] = df['Quantity'].fillna(df['Total Spent'] / df['Price Per Unit'])

#Calcolo di  Total Spent dove ho Price Per Unit e Quantity
df['Total Spent'] = df['Total Spent'].fillna(df['Price Per Unit'] * df['Quantity'])

#Calcolo di Price per Unit dove ho Total Spent e Quantity
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Total Spent'] / df['Quantity'])


In [26]:
#Valori mancanti  
missing= df.isna().sum()
print(missing)



Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit         0
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
Year                   0
Month                  0
Day_name               0
dtype: int64


In questo modo abbiamo trovato solo i valori di 'Price per Unit' , gli altri due valori no perchè i valori mancanti si trovano sulla stessa riga 


**Verifica 2** 

In [27]:
#Creo una mappa e associo ogni (Category, Price per Unit) al suo Item unico

item_mapping=(
    df.dropna(subset=['Item','Category','Price Per Unit']) #Filtro in df tenendo solo le righe in cui le tre colonne hanno tutti i valori (ovvero non sono NaN)
    .groupby(['Category','Price Per Unit'])['Item'] # Raggruppo per 'Category' e 'Price per Unit'e le asseggno a Item
    .first()  #Prendo l articolo associato
    .to_dict()
)

In [28]:
# Creiamo una serie temporanea cercando la combinazione (Category, Price Per Unit) nella mappa
items_calcolati = df.set_index(['Category','Price Per Unit']).index.map(item_mapping)

#Riempiamo i valori mancanti di 'Item'
df['Item']= df['Item'].fillna(pd.Series(items_calcolati, index= df.index))

In [29]:
#Verifica
print("Item mancanti rimasti:", df['Item'].isna().sum())

Item mancanti rimasti: 0


Dato che 'Quantity' e 'Total spent' hanno valori NaN sulla stessa riga, non posso ricavarli matematicamente . Devo quindi decidere se eliminare queste righe e ridurre il mio dataset oppure imputare utilizzando o la medianan  o la moda 

In [30]:
#Imputo 'Quantity usando la mediana per ciascun Item
df['Quantity']= df['Quantity'].fillna(df.groupby('Item')['Quantity'].transform('median'))


In [31]:
#Verifica
print("Quantity rimasti:", df['Quantity'].isna().sum())

Quantity rimasti: 0


In [32]:
#Calcolo, ora che ho sia 'Quantity' che 'Price Per Unit' , la colonna 'Total spent'
df['Total Spent'] = df['Total Spent'].fillna(df['Price Per Unit'] * df['Quantity'])


In [33]:
#Verifica
print("Total Spent mancanti rimasti:", df['Total Spent'].isna().sum())

Total Spent mancanti rimasti: 0


**Verifica 3**

In [36]:
#Creo il 'Total Spent' teorico, senza sconti
Total_Spent_teorico =df['Price Per Unit']* df['Quantity'] 

#Se il Discount Applied è NaN, verifico se c'è stata una differenza di prezzo.
#Se il prezzo reale equivale a quello teorico --> False (Nessuno sconto), altrimenti True(sconto applicato)
df['Discount Applied']= df['Discount Applied'].fillna(df['Total Spent']< Total_Spent_teorico)

In [38]:
#Verifica 
print( 'Discount Applied rimasti:' ,df['Discount Applied'].isna().sum())

Discount Applied rimasti: 0


Verifica finale sui valori nulli (NaN) presenti nel dataset

In [39]:
df.isnull().sum()

Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
Year                0
Month               0
Day_name            0
dtype: int64

Perfetto , il nostro dataset è pulito e pronto per essere analizzato 

In [40]:
#Salvataggio del nostro dataset pulito 
df.to_csv('df_cleaning.csv', index=False)